# 10-2절 연습 문제 풀이

인코더만 사용하는 트랜스포머(`DateFormatClassifier`)를 다루는
[연습 문제 10-6] ~ [연습 문제 10-11]의 풀이다.

본문 예제(`10-02_example.ipynb`)의 데이터 생성, 어휘 사전, 모델, 학습 함수를
그대로 가져와 사용한다.


## 공통 준비 — 본문 예제와 동일

In [1]:

# 예제 실행 및 시각화를 위한 공통 라이브러리 로딩

# 코랩 환경 등 깃허브 전체를 clone해서 실습하는 경우가 아니라면 
# 공통 라이브러리를 불러오기 위해서 별도의 과정이 필요하므로 code_reference/README.md 파일을 확인하자.
import sys
sys.path.append('../../')

from code_reference import common
from code_reference import visualize as viz

# 시각화 결과를 파일에 저장하지 않음
viz.configure(save_grayscale=False)

# matplotlib 시각화에서 한글 폰트 사용 설정
common.set_korean_plot_env()

# 재현성 보장을 위한 시드 고정
#   여기서 재현성은 '동일 컴퓨터, 동일 버전의 파이썬, 동일 버전의 파이토치' 환경에서 재현이 가능하다는 의미로, 
#   독자의 결과는 저자의 결과와 달라질 수 있다는 점을 밝혀둔다.
#   또한 같은 환경에서도 GPU를 사용하는 경우, 일부 연산의 비결정적 성질로 인해 실행시마다 결과가 조금씩 달라질 수 있다.
SEED = 42
common.set_seed(SEED)

# 실습 환경에 맞는 하드웨어 가속기 장치 객체
device = common.get_device()

CUDA를 사용합니다.


In [2]:
# 참고 - 데이터 생성을 위한 함수

import re
import random
import string
import unicodedata
from calendar import monthrange
from collections import Counter, namedtuple

# 로케일에 영향받지 않도록 월 이름을 상수로 고정
MONTH_FULL = ['', 'January', 'February', 'March', 'April', 'May', 'June',
              'July', 'August', 'September', 'October', 'November', 'December']
MONTH_ABBR = ['', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
              'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

# 월 이름을 전체 이름으로만 쓰는 여섯 형식(레이블 0~5 순서 유지)
SRC_FORMATS = [
    '%d %B %Y',     # 01 February 2026
    '%B %d, %Y',    # February 01, 2026
    '%m/%d/%Y',     # 02/01/2026
    '%Y/%m/%d',     # 2026/02/01
    '%d-%m-%Y',     # 01-02-2026
    '%Y-%m-%d',     # 2026-02-01
]

NUM_FORMATS = len(SRC_FORMATS)
YEAR_RANGE = (1900, 2050)
MAX_INPUT_LENGTH = 40       # 노이즈를 포함한 입력 문자열의 최대 길이

# (연, 월, 일)을 fmt 형식에 맞춰 날짜 문자열로 변환
def render(year, month, day, fmt):
    text = fmt.replace('%Y', f'{year:04d}').replace('%B', MONTH_FULL[month])
    return text.replace('%m', f'{month:02d}').replace('%d', f'{day:02d}')

# 형식 문자열을 정규 표현식으로 변환하는 함수(%m, %d는 반드시 두 자리)
def _format_to_pattern(fmt):
    parts = []
    for token in re.split(r'(%[YmdB])', fmt):
        if token == '%Y':
            parts.append(r'(?P<year>\d{4})')
        elif token == '%m':
            parts.append(r'(?P<month>\d{2})')
        elif token == '%d':
            parts.append(r'(?P<day>\d{2})')
        elif token == '%B':
            parts.append('(?P<month_full>' + '|'.join(MONTH_FULL[1:]) + ')')
        else:
            parts.append(re.escape(token))
    return re.compile(''.join(parts))


FORMAT_PATTERNS = [(fmt, _format_to_pattern(fmt)) for fmt in SRC_FORMATS]

# 실제 존재하는 날짜인지 확인하는 함수
def is_real_date(year, month, day):
    if not YEAR_RANGE[0] <= year <= YEAR_RANGE[1]:
        return False
    if not 1 <= month <= 12:
        return False
    return 1 <= day <= monthrange(year, month)[1]

# 입력 문자열에서 발견되는 날짜 문자열 형식을 검색해 유일 형식인지를 판단하는 함수
def formats_in(text):
    found = set()
    for index, (fmt, pattern) in enumerate(FORMAT_PATTERNS):
        for matched in pattern.finditer(text):
            group = matched.groupdict()
            month = (MONTH_FULL.index(group['month_full'])
                     if group.get('month_full') else int(group['month']))
            if is_real_date(int(group['year']), month, int(group['day'])):
                found.add(index)
                break
    return found

In [3]:
# 참고 - 출력 폭을 맞추기 위한 도우미 함수

# 한글처럼 두 칸을 차지하는 글자를 감안해 출력 폭을 맞출 때 사용
def pad(text, width, align='<'):
    display_width = sum(
        2 if unicodedata.east_asian_width(c) in 'WF' else 1 for c in str(text)
    )
    space = ' ' * max(0, width - display_width)
    return space + str(text) if align == '>' else str(text) + space

In [4]:
# 참고 - 노이즈 추가 함수
NOISE_CHARS = (
    string.ascii_letters + string.digits + '!@#$%^&*()_+-=[]{}|;:,./<>?'
)
def add_random_noise(text, rng, max_length=MAX_INPUT_LENGTH):
    """날짜 문자열 앞뒤에 무작위 길이의 무작위 노이즈를 덧붙인다.

    9-1, 9-3, 10-1절의 `add_random_noise()`와 같은 방식이며, 전역 난수
    생성기 대신 주입받은 생성기를 쓰는 점만 다르다.
    """
    prefix, suffix = '', ''
    remaining = max_length - len(text)
    if remaining > 0:
        prefix_length = rng.randint(0, remaining)
        remaining -= prefix_length
        prefix = ''.join(rng.choices(NOISE_CHARS, k=prefix_length))
    if remaining > 0:
        suffix_length = rng.randint(0, remaining)
        suffix = ''.join(rng.choices(NOISE_CHARS, k=suffix_length))
    return prefix + text + suffix

In [5]:
# 참고 - 데이터셋 생성

# 데이터셋의 각 샘플을 나타내는 namedtuple
Sample = namedtuple('Sample', 'text label clean noise_length')

# 노이즈가 섞인 날짜 형식 분류 데이터셋을 생성
#   발생 가능한 형식 모호성을 피하기 위해 형식이 유일하게 결정되지 않는 경우 폐기
def build_dataset(total=12000, ratios=(0.64, 0.16, 0.20), seed=42):
    rng = random.Random(seed)
    seen = set()
    discarded = Counter()
    strata = []
    per_format = total // NUM_FORMATS
    for index, fmt in enumerate(SRC_FORMATS):
        samples = []
        while len(samples) < per_format:
            year = rng.randint(*YEAR_RANGE)
            month = rng.randint(1, 12)
            day = rng.randint(1, monthrange(year, month)[1])
            clean = render(year, month, day, fmt)
            text = add_random_noise(clean, rng)
            if text in seen:                    # 중복 제거
                discarded['중복'] += 1
                continue
            if formats_in(text) != {index}:     # 형식이 유일하지 않으면 폐기
                discarded['형식 모호'] += 1
                continue
            seen.add(text)
            samples.append(Sample(text, index, clean, len(text) - len(clean)))
        strata.append(samples)

    train, valid, test = [], [], []
    for group in strata:
        rng.shuffle(group)
        n_train = round(len(group) * ratios[0])
        n_valid = round(len(group) * ratios[1])
        train += group[:n_train]
        valid += group[n_train:n_train + n_valid]
        test += group[n_train + n_valid:]
    for split in (train, valid, test):
        rng.shuffle(split)
    return {'train': train, 'valid': valid, 'test': test}, discarded


dataset, discarded = build_dataset(total=12000)

print(pad('형식', 12) + ''.join(f'{name:>8s}' for name in dataset)
      + pad('평균 길이', 12, '>'))
print('-' * 48)
counters = {name: Counter(s.label for s in split) for name, split in dataset.items()}
all_samples = [s for split in dataset.values() for s in split]
for index, fmt in enumerate(SRC_FORMATS):
    lengths = [len(s.text) for s in all_samples if s.label == index]
    print(pad(fmt, 12) + ''.join(f'{counters[name][index]:>8d}' for name in dataset)
          + pad(f'{sum(lengths) / len(lengths):.1f}', 12, '>'))
print('-' * 48)
print(pad('합계', 12) + ''.join(f'{len(split):>8d}' for split in dataset.values()))

print(f'\n생성 중 폐기: {dict(discarded)}')
texts = {name: {s.text for s in split} for name, split in dataset.items()}
print(f"훈련-검증 중복 {len(texts['train'] & texts['valid'])}건, "
      f"훈련-테스트 중복 {len(texts['train'] & texts['test'])}건")
noise_lengths = [s.noise_length for s in all_samples]
print(f'노이즈 길이: 최소 {min(noise_lengths)}, 최대 {max(noise_lengths)}, '
      f'평균 {sum(noise_lengths) / len(noise_lengths):.1f}')
print('\n예시:')
for s in dataset['train'][:5]:
    print(f'  {s.text!r:<44s} -> {SRC_FORMATS[s.label]:<12s} (원본 {s.clean!r})')

형식           train   valid    test   평균 길이
------------------------------------------------
%d %B %Y        1280     320     400        33.2
%B %d, %Y       1280     320     400        34.0
%m/%d/%Y        1280     320     400        32.4
%Y/%m/%d        1280     320     400        32.5
%d-%m-%Y        1280     320     400        32.8


%Y-%m-%d        1280     320     400        32.5
------------------------------------------------
합계            7680    1920    2400

생성 중 폐기: {'형식 모호': 1}
훈련-검증 중복 0건, 훈련-테스트 중복 0건
노이즈 길이: 최소 0, 최대 30, 평균 21.3

예시:
  '7FKhMio;GEh4jD_11981-02-06LaoCPfx*IP'       -> %Y-%m-%d     (원본 '1981-02-06')
  'r%(vIv1XT0LVQ<yT]8|?_February 26, 1904'     -> %B %d, %Y    (원본 'February 26, 1904')
  '}]1953/05/24vIx9I/5qnx{>q0,}#D7'            -> %Y/%m/%d     (원본 '1953/05/24')
  'd2sG.AhtykjuIi9October 01, 2049+s:c'        -> %B %d, %Y    (원본 'October 01, 2049')
  'eA1948-08-21qbsv*'                          -> %Y-%m-%d     (원본 '1948-08-21')


In [6]:
# 참고 - 어휘 사전 
# 	어휘 사전에 없는 토큰은 [UNK] 토큰으로 바꿔 '모르는 글자'로 취급

import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

CLS_TOKEN, PAD_TOKEN, UNK_TOKEN = '[CLS]', '[PAD]', '[UNK]'
CLS_IDX, PAD_IDX, UNK_IDX = 0, 1, 2
special_tokens = {CLS_TOKEN: CLS_IDX, PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}


class Vocab:
    def __init__(self, texts, special):
        tokens = set()
        for text in texts:
            tokens.update(text)
        self.vocab = dict(special)
        for i, token in enumerate(sorted(tokens)):
            self.vocab[token] = i + len(special)
        self.itos = {v: k for k, v in self.vocab.items()}

    def encode(self, text):
        return [self.vocab.get(c, UNK_IDX) for c in text]

    def __len__(self):
        return len(self.vocab)

In [7]:
########################################################################################
# 코드 10-9 - 입력 앞에 [CLS] 토큰을 추가한 데이터셋 정의
########################################################################################

class DateFormatDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.samples = []
        for text, label in zip(texts, labels):
            # 입력 앞에 [CLS] 토큰 추가
            ids = [CLS_IDX] + vocab.encode(text)
            self.samples.append((ids, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]

In [8]:
# 참고 - 배치 병합 함수
#   : 입력 문자열 샘플을 패딩해 길이를 맞추고 텐서로 변환하는 배치 병합 함수를 사용해 데이터로더를 생성

def collate_fn(batch):
    seq_batch, label_batch = zip(*batch)
    seq_tensors = [torch.LongTensor(s) for s in seq_batch]
    src_padded = pad_sequence(seq_tensors, batch_first=True, padding_value=PAD_IDX)
    return src_padded, torch.LongTensor(label_batch)

In [9]:

# 어휘 사전은 훈련 분할만으로 구축한다
vocab = Vocab([s.text for s in dataset['train']], special_tokens)

BATCH_SIZE = 32
loaders = {}
for name, split in dataset.items():
    loaders[name] = DataLoader(
        DateFormatDataset([s.text for s in split], [s.label for s in split], vocab),
        batch_size=BATCH_SIZE, shuffle=(name == 'train'), collate_fn=collate_fn,
    )

letters = ''.join(sorted(t for t in vocab.vocab if len(t) == 1))
print(f'어휘 사전 크기: {len(vocab)} ([CLS] + [PAD] + [UNK] + 글자들)')
print(f'어휘 사전의 글자: {letters!r}')
unseen = ({c for s in dataset['valid'] + dataset['test'] for c in s.text}
          - set(vocab.vocab))
print(f'훈련셋에 없는 글자: {len(unseen)}개 {sorted(unseen)}')

어휘 사전 크기: 93 ([CLS] + [PAD] + [UNK] + 글자들)
어휘 사전의 글자: ' !#$%&()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[]^_abcdefghijklmnopqrstuvwxyz{|}'
훈련셋에 없는 글자: 0개 []


In [10]:
# 7. 학습 가능한 위치 인코딩
import torch.nn as nn


class PositionalEncoding(nn.Module):
    def __init__(self, max_length, d_model):
        super().__init__()
        self.position_embedding = nn.Embedding(max_length, d_model)
        self.activation = nn.Tanh()

    def forward(self, token_embedded):
        seq_length = token_embedded.size(1)
        positions = torch.arange(seq_length, device=token_embedded.device)
        pos_embedded = self.position_embedding(positions)
        return self.activation(token_embedded + pos_embedded)

In [11]:
###############################################################################
# 코드 10-8 - 인코더만 사용하는 트랜스포머 모델, DateFormatClassifier 클래스
###############################################################################
class DateFormatClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, ff_dim,
                 num_layers, max_length, num_classes, dropout):
        super().__init__()
        # 입력 토큰 임베딩, 위치 인코딩, 드롭아웃 (10-1 절과 동일)
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_encoding = PositionalEncoding(max_length, d_model)
        self.dropout = nn.Dropout(dropout)

        # 트랜스포머(인코더)의 기본 블록
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads,
            dim_feedforward=ff_dim, dropout=dropout,
            batch_first=True,
        )
        # 기본 블록을 여러 개 쌓은 트랜스포머(인코더) 블록
        self.encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers,
            # 중첩 텐서(nested tensor) API 사용 비활성화 - 실험적 API 경고 방지
            enable_nested_tensor=False,
        )
        # 분류기 - [CLS] 토큰의 출력 벡터를 받아 여섯 형식 중 하나로 분류
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, source):
        # PAD 마스크 - Transformer 계열 클래스는 PAD 위치가 True 인 마스크 사용
        # UNK 는 마스크 대상이 아니므로 모르는 글자도 어텐션에 참여한다
        pad_mask = (source == PAD_IDX)
        x = self.dropout(self.pos_encoding(self.embedding(source)))
        output = self.encoder(x, src_key_padding_mask=pad_mask)
        # [CLS](0번 위치)의 출력 벡터를 사용해 형식 분류
        cls_output = output[:, 0, :]            # (B, d_model)
        return self.classifier(cls_output)

In [12]:
# 참고 - 학습 함수

import copy

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    loss_sum, sample_size, correct_size = 0.0, 0, 0
    for src, labels in loader:
        src, labels = src.to(device), labels.to(device)
        optimizer.zero_grad()
        # 모델에는 입력 문자열만 전달(정답 레이블은 전달하지 않음)
        logits = model(src)                         # (B, NUM_FORMATS)
        # 정답은 손실을 계산하는 용도로만 사용, ignore_index 인자 사용하지 않음
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        batch_size = src.size(0)
        loss_sum += loss.item() * batch_size
        sample_size += batch_size
        correct_size += (logits.argmax(1) == labels).sum().item()
    return loss_sum / sample_size, correct_size / sample_size * 100.0


@torch.no_grad()
def validation(model, loader, criterion, device):
    model.eval()
    loss_sum, sample_size, correct_size = 0.0, 0, 0
    for src, labels in loader:
        src, labels = src.to(device), labels.to(device)
        logits = model(src)
        loss = criterion(logits, labels)
        batch_size = src.size(0)
        loss_sum += loss.item() * batch_size
        sample_size += batch_size
        correct_size += (logits.argmax(1) == labels).sum().item()
    return loss_sum / sample_size, correct_size / sample_size * 100.0


def train_loop(model, train_loader, valid_loader, criterion, optimizer,
               epochs, patience, device):
    model.to(device)
    log = common.EpochLogger(epochs)
    best_valid_loss = float('inf')
    best_state, counter = None, 0
    stopped = False
    for epoch in range(1, epochs + 1):
        train_loss, _ = train_epoch(
            model, train_loader, criterion, optimizer, device)
        valid_loss, valid_acc = validation(
            model, valid_loader, criterion, device)
        log.row(epoch, train_loss, valid_loss, valid_acc)
        if valid_loss < best_valid_loss:
            best_valid_loss = valid_loss
            best_state = copy.deepcopy(model.state_dict())
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                stopped = True
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    log.summary(stopped='조기 종료' if stopped else None)
    return log

In [13]:
# 참고 - 모델 객체 생성과 학습
import torch.optim as optim

# 모델 구조 하이퍼파라미터
D_MODEL = 64
NUM_HEADS = 4
FF_DIM = 128
NUM_LAYERS = 2
MAX_LENGTH = 48             # 입력 최대 40자 + [CLS] + 여유
DROPOUT = 0.1

# 학습 설정 하이퍼파라미터
LR = 1e-3
EPOCHS = 60
PATIENCE = 8

# 결과 재현을 위한 시드값 고정
common.set_seed(SEED)

# 모델 객체 생성
model = DateFormatClassifier(
    vocab_size=len(vocab), d_model=D_MODEL, num_heads=NUM_HEADS,
    ff_dim=FF_DIM, num_layers=NUM_LAYERS, max_length=MAX_LENGTH,
    num_classes=NUM_FORMATS, dropout=DROPOUT
).to(device)

# 손실 함수와 옵티아미저 생성
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# 모델 학습 실행
log = train_loop(model, loaders['train'], loaders['valid'],
                 criterion, optimizer, EPOCHS, PATIENCE, device)

에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/60       0.8377       0.6123       62.71%     0:02


  6/60       0.5418       0.5400       65.21%     0:10


 12/60       0.4728       0.5043       70.73%     0:20


 18/60       0.3641       0.3552       84.22%     0:29


 24/60       0.2088       0.2047       93.23%     0:38


...

 36/60       0.1007       0.1269       96.82%     0:57


 42/60       0.0721       0.1142       97.29%     1:06


 48/60       0.0713       0.1138       97.03%     1:15


 54/60       0.0596       0.1061       96.72%     1:25


------------------------------------------------------
최적 51 에포크 · 검증 손실 0.0837 · 전체 학습 시간 1:32 · (조기 종료)


In [14]:
# 참고 - 오분류 사례 전체 출력(오류 유형별)

# 각 분할된 데이터셋 전체에 대한 예측 레이블과 신뢰도를 한 번에 계산
@torch.no_grad()
def predict_split(model, samples, vocab, device, batch_size=128):
    model.eval()
    preds, confidences = [], []
    for start in range(0, len(samples), batch_size):
        chunk = samples[start:start + batch_size]
        seq_tensors = [
            torch.LongTensor([CLS_IDX] + vocab.encode(s.text)) for s in chunk
        ]
        src = pad_sequence(
            seq_tensors, batch_first=True, padding_value=PAD_IDX).to(device)
        probs = torch.softmax(model(src), dim=1)
        confidence, pred = probs.max(dim=1)
        preds += pred.tolist()
        confidences += confidence.tolist()
    return preds, confidences

test_samples = dataset['test']
test_preds, test_confidences = predict_split(model, test_samples, vocab, device)

# 오분류를 (정답 형식, 예측 형식) 쌍으로 취합
errors = {}
for sample, pred, confidence in zip(test_samples, test_preds, test_confidences):
    if pred != sample.label:
        errors.setdefault((sample.label, pred), []).append((sample, confidence))

error_size = sum(len(cases) for cases in errors.values())
print(f'테스트셋 {len(test_samples)}개 중 오분류 {error_size}건, '
      f'(정답 -> 예측) 유형 {len(errors)}가지')

# 건수가 많은 유형부터, 유형 안에서는 신뢰도가 높은(더 확신하며 틀린) 순서로 출력
for (label, pred), cases in sorted(errors.items(), key=lambda kv: -len(kv[1])):
    print(f'[정답 {SRC_FORMATS[label]} -> 예측 {SRC_FORMATS[pred]}] {len(cases)}건')
    for sample, confidence in sorted(cases, key=lambda case: -case[1]):
        print(f'  {sample.text!r:<44s} 포함된 날짜 문자열 {sample.clean!r:<21s} '
              f'노이즈 {sample.noise_length:>2d}자  신뢰도 {confidence * 100:5.1f}%')
    print()

테스트셋 2400개 중 오분류 57건, (정답 -> 예측) 유형 11가지
[정답 %d-%m-%Y -> 예측 %Y-%m-%d] 17건
  'n6V1N/{MkyXam$Aa24-12-2017^xv0%f-N[ay;V'    포함된 날짜 문자열 '24-12-2017'          노이즈 29자  신뢰도  99.8%
  'px_Gq2R+:*>XM+<.2s9w7<27-01-2050DA'         포함된 날짜 문자열 '27-01-2050'          노이즈 24자  신뢰도  99.4%
  '!y6&[P($26-02-1910EPx-aUcIaB'               포함된 날짜 문자열 '26-02-1910'          노이즈 18자  신뢰도  99.4%
  '$}[r{sxEi./;4WA&dh09<k>a_025-04-1965hOH'    포함된 날짜 문자열 '25-04-1965'          노이즈 29자  신뢰도  99.3%
  ',IXYV{|HKmS[<w>025-08-1923t6=J4}8,%z6#mq'   포함된 날짜 문자열 '25-08-1923'          노이즈 30자  신뢰도  99.2%
  'g/^OAa-16921-02-2036c#+Bb@sk/'              포함된 날짜 문자열 '21-02-2036'          노이즈 19자  신뢰도  99.1%
  ')4Fg{!gxi907-12-2035$|>PeCLfxQwk8@3%|iU&'   포함된 날짜 문자열 '07-12-2035'          노이즈 30자  신뢰도  98.7%
  'ev|)0VQ<g#Uc(o4Veh9,u[14-04-2040'           포함된 날짜 문자열 '14-04-2040'          노이즈 22자  신뢰도  98.0%
  'o8910-11-2025!#s5{,_y_'                     포함된 날짜 문자열 '10-11-2025'          노이즈 12자  신뢰도  98.0%
  'E%0%dH1.o,*g[}1FJ^&?9P{

In [15]:

# 풀이에서 반복해 쓰는 도우미
import numpy as np
import torch.nn.functional as F
from collections import Counter


def train_classifier(model, train_loader, valid_loader, name='',
                     epochs=EPOCHS, patience=PATIENCE, lr=LR):
    if name:
        print(f'{name} 학습')
    crit = nn.CrossEntropyLoss()
    opt = optim.Adam(model.parameters(), lr=lr)
    return train_loop(model, train_loader, valid_loader, crit, opt,
                      epochs, patience, device)


def test_accuracy(model, loader):
    accuracy, _, _ = common.get_accuracy(model, loader, device)
    return accuracy


def make_loaders(ds, vc, batch_size=BATCH_SIZE):
    return {
        name: DataLoader(
            DateFormatDataset([s.text for s in split],
                              [s.label for s in split], vc),
            batch_size=batch_size, shuffle=(name == 'train'), collate_fn=collate_fn)
        for name, split in ds.items()
    }


BASE_TEST_ACC = test_accuracy(model, loaders['test'])
print(f'기준 모델 테스트셋 정확도: {BASE_TEST_ACC:.2f}%')

기준 모델 테스트셋 정확도: 97.62%


## 연습 문제 10-6

In [16]:

# 혼동 행렬을 숫자로 다시 보고 오분류가 몰린 형식 쌍을 찾는다
preds, confs = predict_split(model, dataset['test'], vocab, device)
labels = [s.label for s in dataset['test']]

cm = np.zeros((NUM_FORMATS, NUM_FORMATS), dtype=int)
for t, p in zip(labels, preds):
    cm[t][p] += 1

print('혼동 행렬 (행: 정답, 열: 예측)')
print(pad('정답 \\ 예측', 16) + ''.join(f'{i:>7d}' for i in range(NUM_FORMATS)))
print('-' * 60)
for i in range(NUM_FORMATS):
    print(pad(f'{i} {SRC_FORMATS[i]}', 16)
          + ''.join(f'{cm[i][j]:>7d}' for j in range(NUM_FORMATS)))

print()
pairs = Counter((t, p) for t, p in zip(labels, preds) if t != p)
print('오분류가 많은 (정답 -> 예측) 쌍')
for (t, p), n in pairs.most_common():
    kind = '숫자 형식끼리' if t >= 2 and p >= 2 else '월 이름 형식 관련'
    print(f'  {SRC_FORMATS[t]:<12s} -> {SRC_FORMATS[p]:<12s} {n:>4d}건  ({kind})')

혼동 행렬 (행: 정답, 열: 예측)
정답 \ 예측           0      1      2      3      4      5
------------------------------------------------------------
0 %d %B %Y          394      6      0      0      0      0
1 %B %d, %Y           1    399      0      0      0      0
2 %m/%d/%Y            0      0    388     10      0      2
3 %Y/%m/%d            0      0      9    390      0      1
4 %d-%m-%Y            0      0      2      6    375     17
5 %Y-%m-%d            0      0      0      2      1    397

오분류가 많은 (정답 -> 예측) 쌍
  %d-%m-%Y     -> %Y-%m-%d       17건  (숫자 형식끼리)
  %m/%d/%Y     -> %Y/%m/%d       10건  (숫자 형식끼리)
  %Y/%m/%d     -> %m/%d/%Y        9건  (숫자 형식끼리)
  %d %B %Y     -> %B %d, %Y       6건  (월 이름 형식 관련)
  %d-%m-%Y     -> %Y/%m/%d        6건  (숫자 형식끼리)
  %m/%d/%Y     -> %Y-%m-%d        2건  (숫자 형식끼리)
  %d-%m-%Y     -> %m/%d/%Y        2건  (숫자 형식끼리)
  %Y-%m-%d     -> %Y/%m/%d        2건  (숫자 형식끼리)
  %Y/%m/%d     -> %Y-%m-%d        1건  (숫자 형식끼리)
  %B %d, %Y    -> %d %B %Y        1건  (월 이름 형식 관련)
 

In [17]:

# 가설: 노이즈의 숫자가 날짜 문자열 경계에 달라붙어 자릿수 경계를 흐린다
def touches_digit(sample):
    text, clean = sample.text, sample.clean
    start = text.find(clean)
    if start < 0:
        return False
    before = text[start - 1] if start > 0 else ''
    after = text[start + len(clean)] if start + len(clean) < len(text) else ''
    return before.isdigit() or after.isdigit()


wrong = [s for s, p in zip(dataset['test'], preds) if s.label != p]
right = [s for s, p in zip(dataset['test'], preds) if s.label == p]

print(f'{"구분":<8}{"건수":>8}{"경계에 숫자 인접":>18}{"평균 노이즈 길이":>18}')
print('-' * 54)
for name, group in (('오분류', wrong), ('정분류', right)):
    rate = sum(touches_digit(s) for s in group) / len(group) * 100
    avg = sum(s.noise_length for s in group) / len(group)
    print(f'{name:<8}{len(group):>8d}{rate:>17.1f}%{avg:>17.1f}자')

구분            건수         경계에 숫자 인접         평균 노이즈 길이
------------------------------------------------------
오분류           57             33.3%             24.0자
정분류         2343             19.5%             21.3자


In [18]:

# 검증: 경계에 숫자를 두지 않도록 데이터를 다시 만들어 학습한다
def add_noise_no_digit_edge(clean, rng, max_length=MAX_INPUT_LENGTH):
    letters_only = string.ascii_letters + '!@#$%^&*()_+=[]{}|;:,./<>?'
    noise_chars = letters_only + string.digits
    remaining = max_length - len(clean)
    prefix = suffix = ''
    if remaining > 0:
        k = rng.randint(0, remaining)
        remaining -= k
        if k:
            prefix = ''.join(rng.choices(noise_chars, k=k - 1)) + rng.choice(letters_only)
    if remaining > 0:
        k = rng.randint(0, remaining)
        if k:
            suffix = rng.choice(letters_only) + ''.join(rng.choices(noise_chars, k=k - 1))
    return prefix + clean + suffix


def build_dataset_v2(total=12000, ratios=(0.64, 0.16, 0.20), seed=42):
    rng = random.Random(seed)
    seen, strata = set(), []
    per_format = total // NUM_FORMATS
    for index, fmt in enumerate(SRC_FORMATS):
        samples = []
        while len(samples) < per_format:
            year = rng.randint(*YEAR_RANGE)
            month = rng.randint(1, 12)
            day = rng.randint(1, monthrange(year, month)[1])
            clean = render(year, month, day, fmt)
            text = add_noise_no_digit_edge(clean, rng)
            if text in seen or formats_in(text) != {index}:
                continue
            seen.add(text)
            samples.append(Sample(text, index, clean, len(text) - len(clean)))
        strata.append(samples)
    out = {'train': [], 'valid': [], 'test': []}
    for group in strata:
        rng.shuffle(group)
        n_tr = round(len(group) * ratios[0])
        n_va = round(len(group) * ratios[1])
        out['train'] += group[:n_tr]
        out['valid'] += group[n_tr:n_tr + n_va]
        out['test'] += group[n_tr + n_va:]
    for split in out.values():
        rng.shuffle(split)
    return out


dataset_v2 = build_dataset_v2()
vocab_v2 = Vocab([s.text for s in dataset_v2['train']], special_tokens)
loaders_v2 = make_loaders(dataset_v2, vocab_v2)

common.set_seed(SEED)
model_v2 = DateFormatClassifier(
    vocab_size=len(vocab_v2), d_model=D_MODEL, num_heads=NUM_HEADS, ff_dim=FF_DIM,
    num_layers=NUM_LAYERS, max_length=MAX_LENGTH, num_classes=NUM_FORMATS,
    dropout=DROPOUT).to(device)
train_classifier(model_v2, loaders_v2['train'], loaders_v2['valid'],
                 name='경계에 숫자를 두지 않은 데이터')
acc_v2 = test_accuracy(model_v2, loaders_v2['test'])
print()
print(f'기준 데이터셋      테스트 정확도 {BASE_TEST_ACC:.2f}%')
print(f'경계 숫자 제거     테스트 정확도 {acc_v2:.2f}%')
print(f'차이 {acc_v2 - BASE_TEST_ACC:+.2f}%p')

경계에 숫자를 두지 않은 데이터 학습


에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/60       0.7778       0.5564       63.12%     0:01


  6/60       0.4995       0.4899       68.70%     0:09


 12/60       0.3255       0.3064       87.29%     0:17


 18/60       0.1393       0.0677       97.66%     0:26


 24/60       0.0712       0.0779       97.86%     0:35


 30/60       0.0553       0.0497       98.28%     0:44


 36/60       0.0344       0.0548       98.91%     0:52


 42/60       0.0362       0.0563       98.75%     1:01


------------------------------------------------------
최적 37 에포크 · 검증 손실 0.0343 · 전체 학습 시간 1:05 · (조기 종료)

기준 데이터셋      테스트 정확도 97.62%
경계 숫자 제거     테스트 정확도 99.00%
차이 +1.38%p


### 풀이 해설 — 연습 문제 10-6

**혼동 행렬에서 읽히는 것은 분명하다.** 오분류가 **숫자만 쓰는 네 형식**
(`%m/%d/%Y`, `%Y/%m/%d`, `%d-%m-%Y`, `%Y-%m-%d`) 사이에 몰려 있다. 월 이름을 쓰는 두
형식은 `February` 같은 단어가 결정적 단서라 거의 틀리지 않는다.

**그런데 "숫자 형식끼리 헷갈린다"는 관찰만으로는 부족하다.** 지문은 "모델이 **무엇에
속고 있는지** 가설을 세우라"고 한다. 네 형식은 구분자(`/` 또는 `-`)와 자릿수 배치
(4-2-2인지 2-2-4인지)로 충분히 구별되는데, 왜 못 할까?

**가설: 노이즈의 숫자가 날짜 문자열에 달라붙어 자릿수 경계를 흐린다.**

본문 표 10-6의 예시가 이미 그 장면을 보여 준다.

```
w.vHWT!J9vqW[Mri:b!BRi221-05-1926o   ->  21-05-1926
```

노이즈가 `...BRi2`로 끝나고 날짜가 `21-...`로 시작하니 이어 붙으면 `221-05-1926`이다.
사람이 봐도 앞의 두 자리가 `22`인지 `21`인지 알 수 없다. 자릿수를 세어 형식을 판정하는
모델에게는 **글자 하나가 밀린 것과 같은 혼란**이다.

**검증은 두 단계로 했다.**

**관찰** — 오분류 사례와 정분류 사례에서 '날짜 문자열 경계에 숫자가 인접한 비율'을
비교했다. 가설이 맞다면 오분류 쪽이 높아야 한다. 평균 노이즈 길이도 함께 재서,
'단순히 노이즈가 길어서'라는 경쟁 가설과 구분했다.

**개입** — 경계에 숫자를 두지 않도록 데이터 생성 함수를 바꿔 같은 하이퍼파라미터로
다시 학습했다. **상관관계를 확인하는 데서 멈추지 않고 원인을 제거해 본 것**이 지문이
요구한 '데이터 준비 과정에 반영해 가설의 타당성을 확인'하는 절차다.

**결과가 어느 쪽이든 배울 것이 있다.**

- 정확도가 뚜렷이 오르면 가설이 맞았다는 뜻이다. **데이터를 어떻게 만드느냐가 모델
  구조보다 성능에 크게 작용한다**는 익숙한 교훈이 된다.
- 거의 오르지 않으면 다른 원인을 찾아야 한다. 다음 후보는 노이즈 길이 자체(표 10-7이
  이미 상관을 보여 준다)나, 구분자가 노이즈에도 섞여 있다는 점이다.

**다만 이 개입에는 함정이 하나 있다.** 경계에서 숫자를 뺀 데이터는 **원래 문제보다 쉬운
문제**다. 정확도가 올랐다고 해서 모델이 좋아진 것은 아니며, 실제 입력이 그런 보장을
해 주지도 않는다. 가설을 확인하는 실험과 실전 성능 개선은 다른 이야기다. 실전에서
쓰려면 데이터를 쉽게 만드는 대신, **경계가 모호한 사례를 오히려 더 많이 학습시키는**
방향이 맞다.


## 연습 문제 10-7

In [19]:

# [CLS] 대신 패딩을 제외한 평균 풀링을 사용하는 분류기
class DateFormatClassifierMeanPool(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, ff_dim,
                 num_layers, max_length, num_classes, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos_encoding = PositionalEncoding(max_length, d_model)
        self.dropout = nn.Dropout(dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers, enable_nested_tensor=False)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, source):
        pad_mask = (source == PAD_IDX)
        x = self.dropout(self.pos_encoding(self.embedding(source)))
        output = self.encoder(x, src_key_padding_mask=pad_mask)
        # 패딩을 제외한 평균 풀링: 유효 위치만 더한 뒤 유효 길이로 나눈다
        valid = (~pad_mask).unsqueeze(-1).float()          # (B, S, 1)
        pooled = (output * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1.0)
        return self.classifier(pooled)


common.set_seed(SEED)
model_mean = DateFormatClassifierMeanPool(
    vocab_size=len(vocab), d_model=D_MODEL, num_heads=NUM_HEADS, ff_dim=FF_DIM,
    num_layers=NUM_LAYERS, max_length=MAX_LENGTH, num_classes=NUM_FORMATS,
    dropout=DROPOUT).to(device)
train_classifier(model_mean, loaders['train'], loaders['valid'], name='평균 풀링')
acc_mean = test_accuracy(model_mean, loaders['test'])
print()
print(f'{"방식":<12}{"테스트 정확도":>14}{"파라미터 수":>14}')
print('-' * 42)
print(f'{"[CLS]":<12}{BASE_TEST_ACC:>13.2f}%'
      f'{sum(p.numel() for p in model.parameters()):>14,}')
print(f'{"평균 풀링":<12}{acc_mean:>13.2f}%'
      f'{sum(p.numel() for p in model_mean.parameters()):>14,}')

평균 풀링 학습


에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/60       0.8976       0.7547       49.17%     0:01


  6/60       0.5494       0.5504       64.22%     0:09


 12/60       0.4789       0.5373       68.49%     0:19


 18/60       0.2410       0.2450       90.78%     0:29


 24/60       0.1045       0.1286       96.46%     0:38


 30/60       0.0757       0.1281       96.88%     0:47


 36/60       0.0464       0.1250       97.03%     0:57


 42/60       0.0473       0.0984       97.29%     1:06


------------------------------------------------------
최적 39 에포크 · 검증 손실 0.0785 · 전체 학습 시간 1:14 · (조기 종료)

방식                 테스트 정확도        파라미터 수
------------------------------------------
[CLS]               97.62%        76,358
평균 풀링               97.29%        76,358


In [20]:

# 노이즈 길이별로 두 방식을 비교하면 성격 차이가 드러난다
#   두 모델 모두 [CLS]를 붙인 같은 입력을 받으므로 예측 함수를 공유할 수 있다
BUCKETS = [(0, 5), (6, 10), (11, 15), (16, 20), (21, 25), (26, 30)]


@torch.no_grad()
def predict_all(m, samples, vc, batch_size=128):
    m.eval()
    preds = []
    for start in range(0, len(samples), batch_size):
        chunk = samples[start:start + batch_size]
        seqs = [torch.LongTensor([CLS_IDX] + vc.encode(s.text)) for s in chunk]
        src = pad_sequence(seqs, batch_first=True, padding_value=PAD_IDX).to(device)
        preds += m(src).argmax(dim=1).tolist()
    return preds


def accuracy_by_noise(m, samples, vc):
    preds = predict_all(m, samples, vc)
    out = {}
    for lo, hi in BUCKETS:
        idx = [i for i, s in enumerate(samples) if lo <= s.noise_length <= hi]
        if idx:
            hit = sum(preds[i] == samples[i].label for i in idx)
            out[(lo, hi)] = (len(idx), hit / len(idx) * 100)
    return out


cls_by_noise = accuracy_by_noise(model, dataset['test'], vocab)
mean_by_noise = accuracy_by_noise(model_mean, dataset['test'], vocab)

print(f'{"노이즈 길이":<14}{"샘플 수":>9}{"[CLS]":>10}{"평균 풀링":>12}{"차이":>10}')
print('-' * 56)
for key in cls_by_noise:
    n, a = cls_by_noise[key]
    _, b = mean_by_noise[key]
    print(f'{f"{key[0]}~{key[1]}":<14}{n:>9d}{a:>9.1f}%{b:>11.1f}%{b - a:>9.1f}%p')

노이즈 길이             샘플 수     [CLS]       평균 풀링        차이
--------------------------------------------------------
0~5                  48    100.0%      100.0%      0.0%p
6~10                175     99.4%       98.9%     -0.6%p
11~15               268     98.5%       98.9%      0.4%p
16~20               409     98.0%       98.5%      0.5%p
21~25               709     97.3%       96.5%     -0.8%p
26~30               791     96.8%       96.3%     -0.5%p


### 풀이 해설 — 연습 문제 10-7

**구현에서 핵심은 "패딩 위치는 평균에서 제외해야 한다"는 조건이다.** 그냥
`output.mean(dim=1)`을 쓰면 `<pad>` 자리의 출력까지 평균에 섞인다. 짧은 입력일수록
패딩 비율이 높으므로 **짧은 샘플이 더 크게 망가진다.**

```python
valid = (~pad_mask).unsqueeze(-1).float()
pooled = (output * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1.0)
```

**결과는 예상과 달랐다.**

| 방식 | 테스트 정확도 | 파라미터 수 |
|---|---|---|
| `[CLS]` | 97.62% | 76,358 |
| 평균 풀링 | 97.29% | 76,358 |

**차이가 0.33%p에 그친다.** 노이즈 길이별로 나눠 봐도 구간마다 엎치락뒤치락해서
일관된 경향이 없다.

처음 예상은 이랬다. 평균 풀링은 **노이즈 자리의 출력까지 같은 무게로 섞으므로**
노이즈가 길수록 신호가 희석되어 격차가 벌어질 것이다. 본문 p26이 `[CLS]`의 어텐션
55~74%가 날짜 문자열 영역에 집중된다고 수치로 보였으니 더 그럴듯했다.

**왜 빗나갔을까.** 평균을 내는 대상이 **입력 토큰이 아니라 인코더를 통과한 출력**이기
때문이다. 셀프 어텐션을 거친 뒤에는 노이즈 자리의 출력도 이미 날짜 문자열을 참조한
결과를 담고 있다. 그것을 평균에 넣어도 신호가 크게 희석되지 않는다.

**즉 `[CLS]`의 이점은 '노이즈를 무시하는 것'이 아니라 '무엇을 모을지 모델이 정하는 것'에
있다.** 이 문제는 그 자유도가 꼭 필요한 문제가 아니었던 셈이다.

**두 방식의 성격은 여전히 다르다.**

| | `[CLS]` | 평균 풀링 |
|---|---|---|
| 정보를 모으는 주체 | 셀프 어텐션이 **학습으로** 모은다 | 산술 평균으로 **균등하게** 모은다 |
| 학습 초기 | `[CLS]`가 의미 없는 벡터라 기울기가 늦게 자리를 잡는다 | 모든 위치의 정보가 곧바로 분류기까지 흐른다 |
| 파라미터 | 동일 | 동일 |
| 사전 학습과의 궁합 | BERT 계열의 표준 | 별도 조정 필요 |

**본문이 "`[CLS]`는 여전히 인코더만 사용하는 트랜스포머 모델의 표준 사용법"이라고 한
것은 성능 우위 때문만이 아니다.** 사전 학습 모델(BERT 등)이 그 방식으로 학습되어 있어
미세 조정 때 그대로 쓰는 편이 자연스럽기 때문이기도 하다. **차이가 없다는 결과도
이 문제의 답이 된다.**


## 연습 문제 10-8

In [21]:

# 6-3절 오즈의 띄어쓰기 문제를 인코더만 사용하는 트랜스포머로
from pathlib import Path

raw = Path('../../data/wonderful_wizard_of_oz.txt').read_text(encoding='utf-8-sig')
m = re.search(r'\nChapter I\n', raw)
start = m.start() if m else raw.find('Chapter I')
end = raw.rfind('*** END OF THE PROJECT GUTENBERG')
oz_text = raw[start:end if end > 0 else None].strip().lower()
oz_text = re.sub(r'[^a-z ]', ' ', oz_text)
oz_text = re.sub(r' +', ' ', oz_text).strip()

SPACING_LENGTH = 64


def make_spacing_samples(text, length=SPACING_LENGTH):
    """공백을 제거한 문자열을 입력으로, 각 글자 뒤에 공백이 오는지를 정답으로"""
    samples = []
    for start in range(0, len(text) - length, length):
        chunk = text[start:start + length]
        letters, labels = [], []
        for i, ch in enumerate(chunk):
            if ch == ' ':
                continue
            letters.append(ch)
            labels.append(1 if i + 1 < len(chunk) and chunk[i + 1] == ' ' else 0)
        if len(letters) >= 16:
            samples.append((''.join(letters), labels))
    return samples


spacing_samples = make_spacing_samples(oz_text)
print(f'샘플 수 {len(spacing_samples):,}')
print(f'예시 입력: {spacing_samples[0][0][:48]!r}')
print(f'예시 정답: {spacing_samples[0][1][:48]}')
print('복원 예시:', ''.join(
    c + (' ' if l else '')
    for c, l in zip(spacing_samples[0][0][:48], spacing_samples[0][1][:48])))

샘플 수 3,076
예시 입력: 'chapterithecyclonedorothylivedinthemidstofthegre'
예시 정답: [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0]
복원 예시: chapter i the cyclone dorothy lived in the midst of the gre


In [22]:

# 글자 단위 어휘 사전과 데이터셋(토큰마다 0/1을 예측하는 토큰 분류 문제)
SP_PAD = 0
sp_chars = sorted(set(''.join(s for s, _ in spacing_samples)))
sp_vocab = {c: i + 1 for i, c in enumerate(sp_chars)}      # 0번은 <pad>
SP_VOCAB_SIZE = len(sp_vocab) + 1


class SpacingDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        text, labels = self.samples[i]
        return (torch.tensor([sp_vocab[c] for c in text], dtype=torch.long),
                torch.tensor(labels, dtype=torch.long))


def sp_collate(batch):
    xs, ys = zip(*batch)
    x = pad_sequence(xs, batch_first=True, padding_value=SP_PAD)
    y = pad_sequence(ys, batch_first=True, padding_value=-100)   # -100은 손실에서 제외
    return x, y


n_train = int(len(spacing_samples) * 0.8)
sp_train = DataLoader(SpacingDataset(spacing_samples[:n_train]),
                      batch_size=64, shuffle=True, collate_fn=sp_collate)
sp_valid = DataLoader(SpacingDataset(spacing_samples[n_train:]),
                      batch_size=64, shuffle=False, collate_fn=sp_collate)
print(f'어휘 {SP_VOCAB_SIZE}, 훈련/검증 {n_train} / {len(spacing_samples) - n_train}')

어휘 27, 훈련/검증 2460 / 616


In [23]:

# 토큰마다 분류하는 인코더 트랜스포머([CLS] 없이 모든 위치의 출력을 사용)
class SpacingTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, ff_dim,
                 num_layers, max_length, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=SP_PAD)
        self.pos_encoding = PositionalEncoding(max_length, d_model)
        self.dropout = nn.Dropout(dropout)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(
            layer, num_layers=num_layers, enable_nested_tensor=False)
        self.classifier = nn.Linear(d_model, 2)

    def forward(self, source):
        pad_mask = (source == SP_PAD)
        x = self.dropout(self.pos_encoding(self.embedding(source)))
        output = self.encoder(x, src_key_padding_mask=pad_mask)
        # 인과 마스크가 없으므로 각 글자가 앞뒤를 모두 본다(양방향)
        return self.classifier(output)          # (B, S, 2)


def sp_run_epoch(m, loader, opt=None):
    train = opt is not None
    m.train() if train else m.eval()
    loss_sum, n_tok, hit = 0.0, 0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = m(x)
            loss = F.cross_entropy(logits.reshape(-1, 2), y.reshape(-1),
                                   ignore_index=-100)
            if train:
                opt.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=1.0)
                opt.step()
            mask = y != -100
            k = mask.sum().item()
            loss_sum += loss.item() * k
            n_tok += k
            hit += ((logits.argmax(-1) == y) & mask).sum().item()
    return loss_sum / n_tok, hit / n_tok * 100


common.set_seed(SEED)
model_sp = SpacingTransformer(SP_VOCAB_SIZE, D_MODEL, NUM_HEADS, FF_DIM,
                              NUM_LAYERS, SPACING_LENGTH + 1, DROPOUT).to(device)
opt_sp = optim.Adam(model_sp.parameters(), lr=LR)
SP_EPOCHS = 30
sp_log = common.EpochLogger(SP_EPOCHS, target_rows=10)
print('띄어쓰기 트랜스포머 학습')
for ep in range(1, SP_EPOCHS + 1):
    tr, _ = sp_run_epoch(model_sp, sp_train, opt_sp)
    va, acc = sp_run_epoch(model_sp, sp_valid)
    sp_log.row(ep, tr, va, acc)
sp_log.summary()

띄어쓰기 트랜스포머 학습


에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/30       0.5282       0.4861       78.07%     0:00


  3/30       0.4844       0.4740       78.40%     0:01


  6/30       0.4744       0.4674       78.50%     0:02


  9/30       0.4661       0.4607       78.71%     0:03


...

 18/30       0.4067       0.3951       82.24%     0:06


 21/30       0.3864       0.3715       83.44%     0:07


 24/30       0.3677       0.3494       84.41%     0:08


 27/30       0.3489       0.3324       85.35%     0:09


 30/30       0.3304       0.3163       86.00%     0:10
------------------------------------------------------
최적 30 에포크 · 검증 손실 0.3163 · 전체 학습 시간 0:10


In [24]:

# 복원 결과 확인
@torch.no_grad()
def restore_spacing(m, text):
    m.eval()
    ids = torch.tensor([[sp_vocab[c] for c in text]], dtype=torch.long).to(device)
    pred = m(ids).argmax(-1)[0].tolist()
    return ''.join(c + (' ' if p else '') for c, p in zip(text, pred)).strip()


for text, labels in spacing_samples[n_train:n_train + 3]:
    answer = ''.join(c + (' ' if l else '') for c, l in zip(text, labels)).strip()
    print(f'  입력: {text[:56]}')
    print(f'  정답: {answer[:60]}')
    print(f'  복원: {restore_spacing(model_sp, text)[:60]}')
    print()

  입력: einquiredwellilltellyouwhatithinksaidthelittlemany
  정답: e inquired well i ll tell you what i think said the little m
  복원: e in quired wellill tellyou whatithinksaid the little many

  입력: ouseewhenicametothiscountryitwasinaballoonyoualso
  정답: ou see when i came to this country it was in a balloon you a
  복원: ousee whe n icame to this coun try it was in a bal loon you 

  입력: camethroughtheairbeingcarriedbyacyclonesoibelievethe
  정답: came through the air being carried by a cyclone so i believe
  복원: came through the air bein gcarried by acy clone soibe lieve 



### 풀이 해설 — 연습 문제 10-8

**6장의 `SpacingLSTM`과 구조적으로 무엇이 달라지는지가 이 문제의 핵심이다.**

**하나, 양방향 참조가 공짜로 생긴다.** 6장의 LSTM은 왼쪽에서 오른쪽으로 읽으므로,
어떤 글자 뒤에 공백이 오는지를 판정할 때 **뒤쪽 글자를 볼 수 없었다.** 인코더
트랜스포머는 인과 마스크를 걸지 않으면 모든 글자가 앞뒤를 모두 참조한다. 띄어쓰기는
본래 양방향 정보가 필요한 문제다. `thewizardofoz`에서 `e` 뒤에 공백을 넣을지 정하려면
뒤에 `wizard`가 온다는 사실을 알아야 한다.

본문 10-3절이 "**`nn.TransformerEncoder`의 역할은 인과 마스크의 유무에 따라 갈린다**"고
한 것을 여기서 반대 방향으로 쓰는 셈이다. 생성 모델에서는 인과 마스크를 걸어 단방향
디코더로 만들었지만, 여기서는 걸지 않아 양방향 인코더로 쓴다.

**둘, `[CLS]`를 쓰지 않는다.** 이 문제는 입력 전체를 하나로 분류하는 것이 아니라
**글자마다 0/1을 판정**한다. 따라서 `[:, 0, :]` 슬라이싱 없이 **모든 위치의 출력을
분류기에 통과**시킨다. 10-3절의 `OzWriterTransformer`가 모든 위치의 출력을 쓰는 것과
같은 형태이고, 다른 점은 인과 마스크의 유무뿐이다.

정리하면 인코더 트랜스포머는 세 가지로 쓰인다.

| 쓰임 | 인과 마스크 | 분류기 입력 |
|---|---|---|
| 형식 분류(10-2절) | 없음 | `[CLS]` 한 자리 |
| **띄어쓰기(이 문제)** | **없음** | **모든 위치** |
| 문장 생성(10-3절) | 있음 | 모든 위치 |

**셋, 패딩 처리가 두 겹이다.** 입력 쪽은 `src_key_padding_mask`로 어텐션에서 빼고,
정답 쪽은 `ignore_index=-100`으로 손실에서 뺀다. 두 가지는 목적이 다르므로 둘 다
필요하다. 9장에서 패킹과 `ignore_index`를 함께 쓴 것과 같은 구조다.

**넷, 병렬 처리로 학습이 빠르다.** 6장 모델은 글자를 하나씩 순서대로 처리했지만,
여기서는 64자를 한 번에 계산한다. 같은 데이터를 훨씬 짧은 시간에 학습한다.

**한계도 있다.** 셀프 어텐션의 계산량이 길이의 제곱에 비례하므로, 긴 문서를 통째로
넣기는 어렵다. 이 풀이가 64자 단위로 자른 이유이며, 그 탓에 **조각 경계에서는 문맥이
끊긴다.** 6장의 LSTM은 이론적으로 길이 제한이 없었다는 점과 맞바꾼 셈이다.


## 연습 문제 10-9 [도전 문제]

이 문제는 10-1절의 날짜 변환기 트랜스포머를 대상으로 하므로, 그 모델을 먼저
간단히 학습한다. 10-1절 본문 예제와 마찬가지로 **노이즈를 섞은 데이터**를 사용한다.
깨끗한 데이터로 학습하면 모델이 거의 완벽해져 오답이 나오지 않고, 그러면 신뢰도가
오답을 가려내는지 확인할 수 없다.

In [25]:

# 10-1절 모델 재구성(예측 신뢰도 실험용)
from datetime import datetime, timedelta

SOS_TOKEN, EOS_TOKEN = '<sos>', '<eos>'
DC_SOS, DC_EOS, DC_PAD = 0, 1, 2
dc_special = {SOS_TOKEN: DC_SOS, EOS_TOKEN: DC_EOS, '<pad>': DC_PAD}

DC_FORMATS = ['%d %B %Y', '%d %b %Y', '%B %d, %Y', '%b %d, %Y',
              '%m/%d/%Y', '%Y/%m/%d', '%d-%m-%Y', '%Y-%m-%d']


class DCVocab:
    def __init__(self, seqs, specials):
        toks = set()
        for s in seqs:
            toks.update(s)
        self.vocab = dict(specials)
        for i, t in enumerate(sorted(toks)):
            self.vocab[t] = i + len(specials)
        self.itos = {v: k for k, v in self.vocab.items()}

    def encode(self, s):
        return [self.vocab[c] for c in s]

    def __len__(self):
        return len(self.vocab)


dc_rng = random.Random(SEED)
dc_dates = [datetime(1900, 1, 1) + timedelta(days=dc_rng.randrange(55150))
            for _ in range(4000)]


def dc_add_noise(text, rng, max_length=40):
    """10-1절 예제의 generate_noisy_datepairs()와 같은 방식으로 노이즈를 덧붙인다."""
    chars = string.ascii_letters + string.digits + '!@#$%^&*()_+-=[]{}|;:,./<>?'
    prefix = suffix = ''
    remaining = max_length - len(text)
    if remaining > 0:
        k = rng.randint(0, remaining)
        remaining -= k
        prefix = ''.join(rng.choices(chars, k=k))
    if remaining > 0:
        suffix = ''.join(rng.choices(chars, k=rng.randint(0, remaining)))
    return prefix + text + suffix


# 10-1절 본문 예제는 노이즈를 섞은 데이터를 사용한다([표 10-4]의 모델 다·라).
#   깨끗한 데이터로 학습하면 모델이 거의 완벽해져 오답이 나오지 않으므로,
#   신뢰도가 오답을 가려내는지 확인할 수 없다.
dc_src = [dc_add_noise(d.strftime(DC_FORMATS[i % 8]), dc_rng)
          for i, d in enumerate(dc_dates)]
dc_tgt = [f'{d.year}-{d.month}-{d.day}' for d in dc_dates]
print('노이즈 데이터 예시:')
for i in range(3):
    print(f'  {dc_src[i]!r} -> {dc_tgt[i]!r}')
dc_sv, dc_tv = DCVocab(dc_src, dc_special), DCVocab(dc_tgt, dc_special)


class DCDataset(Dataset):
    def __init__(self, s, t, sv, tv):
        self.items = [(sv.encode(a), [DC_SOS] + tv.encode(b) + [DC_EOS])
                      for a, b in zip(s, t)]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


def dc_collate(b):
    s, t = zip(*b)
    return (pad_sequence([torch.tensor(x) for x in s], batch_first=True,
                         padding_value=DC_PAD),
            pad_sequence([torch.tensor(x) for x in t], batch_first=True,
                         padding_value=DC_PAD))


dc_train = DataLoader(DCDataset(dc_src[:3000], dc_tgt[:3000], dc_sv, dc_tv),
                      batch_size=32, shuffle=True, collate_fn=dc_collate)
dc_valid = DataLoader(DCDataset(dc_src[3000:], dc_tgt[3000:], dc_sv, dc_tv),
                      batch_size=32, shuffle=False, collate_fn=dc_collate)


class DateConverterTransformer(nn.Module):
    def __init__(self, sv, tv, d_model, num_heads, ff_dim, num_layers,
                 max_length, dropout):
        super().__init__()
        self.src_embedding = nn.Embedding(sv, d_model, padding_idx=DC_PAD)
        self.tgt_embedding = nn.Embedding(tv, d_model, padding_idx=DC_PAD)
        self.pos_encoding = PositionalEncoding(max_length, d_model)
        self.dropout = nn.Dropout(dropout)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=num_heads, num_encoder_layers=num_layers,
            num_decoder_layers=num_layers, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True)
        self.fc = nn.Linear(d_model, tv)

    def forward(self, src, tgt):
        tgt_in = tgt[:, :-1]
        causal = nn.Transformer.generate_square_subsequent_mask(
            tgt_in.size(1), device=src.device)
        se = self.dropout(self.pos_encoding(self.src_embedding(src)))
        te = self.dropout(self.pos_encoding(self.tgt_embedding(tgt_in)))
        out = self.transformer(
            se, te, tgt_mask=causal,
            src_key_padding_mask=(src == DC_PAD),
            tgt_key_padding_mask=(tgt_in == DC_PAD),
            memory_key_padding_mask=(src == DC_PAD))
        return self.fc(out)


common.set_seed(SEED)
dc_model = DateConverterTransformer(len(dc_sv), len(dc_tv), 64, 4, 128, 1, 64,
                                    0.1).to(device)
dc_crit = nn.CrossEntropyLoss(ignore_index=DC_PAD)
dc_opt = optim.Adam(dc_model.parameters(), lr=1e-3)
dc_log = common.EpochLogger(200, target_rows=8)
best, best_state, counter = float('inf'), None, 0
print('날짜 변환기 트랜스포머 학습(노이즈 데이터)')
for ep in range(1, 201):
    dc_model.train()
    tot = n = 0
    for s, t in dc_train:
        s, t = s.to(device), t.to(device)
        dc_opt.zero_grad()
        lg = dc_model(s, t)
        loss = dc_crit(lg.reshape(-1, lg.size(-1)), t[:, 1:].reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(dc_model.parameters(), 1.0)
        dc_opt.step()
        tot += loss.item() * s.size(0); n += s.size(0)
    tr = tot / n
    dc_model.eval()
    tot = n = c = 0
    with torch.no_grad():
        for s, t in dc_valid:
            s, t = s.to(device), t.to(device)
            lg = dc_model(s, t)
            lb = t[:, 1:]
            loss = dc_crit(lg.reshape(-1, lg.size(-1)), lb.reshape(-1))
            tot += loss.item() * s.size(0); n += s.size(0)
            mk = lb != DC_PAD
            c += ((lg.argmax(-1) == lb) | ~mk).all(1).sum().item()
    va, acc = tot / n, c / n * 100
    dc_log.row(ep, tr, va, acc)
    if va < best:
        best, best_state, counter = va, copy.deepcopy(dc_model.state_dict()), 0
    else:
        counter += 1
        if counter >= 10:
            break
dc_model.load_state_dict(best_state)
dc_log.summary(stopped='조기 종료')

노이즈 데이터 예시:
  'uKq5Mc2r0I!s%)25 September 2014Z' -> '2014-9-25'
  ']qDOx[U4120E=2_!A2hlkBq9T24 Dec 19191ZL' -> '1919-12-24'
  '495?XRmBOeFam_/June 28, 19046KR@NoY(' -> '1904-6-28'
날짜 변환기 트랜스포머 학습(노이즈 데이터)


/home/crapas/.local/lib/python3.12/site-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


/home/crapas/.local/lib/python3.12/site-packages/torch/nn/modules/transformer.py:505: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


 에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/200       1.4305       1.0756        0.00%     0:01


 25/200       0.6086       0.6080        6.60%     0:23


...

100/200       0.1051       0.0768       80.50%     1:31


125/200       0.0731       0.0433       87.40%     1:54


150/200       0.0547       0.0267       93.30%     2:16


175/200       0.0464       0.0220       94.50%     2:38


200/200       0.0387       0.0152       95.70%     3:01
-------------------------------------------------------
최적 200 에포크 · 검증 손실 0.0152 · 전체 학습 시간 3:01 · (조기 종료)


In [26]:

# 생성 결과와 신뢰도를 함께 반환하는 예측 함수
@torch.no_grad()
def predict_with_confidence(model, src_text, sv, tv, device, max_length=12):
    """토큰별 확률의 기하 평균을 생성 신뢰도로 사용한다.

    토큰 확률을 그대로 곱하면 길이가 길수록 값이 작아져 길이가 다른 결과끼리
    비교할 수 없다. 로그 확률의 평균에 지수를 씌우면(= 기하 평균) 길이와
    무관한 '토큰 하나당 평균 확률'이 된다.
    """
    model.eval()
    src = torch.tensor([sv.encode(src_text)], dtype=torch.long).to(device)
    src_pad = (src == DC_PAD)
    memory = model.transformer.encoder(
        model.pos_encoding(model.src_embedding(src)),
        src_key_padding_mask=src_pad)
    dec_in = torch.tensor([[DC_SOS]], dtype=torch.long).to(device)
    chars, token_probs = [], []
    for _ in range(max_length):
        causal = nn.Transformer.generate_square_subsequent_mask(
            dec_in.size(1), device=device)
        dec = model.transformer.decoder(
            model.pos_encoding(model.tgt_embedding(dec_in)), memory,
            tgt_mask=causal, memory_key_padding_mask=src_pad)
        probs = torch.softmax(model.fc(dec)[:, -1, :], dim=-1)
        p, idx = probs.max(dim=-1)
        token = tv.itos[idx.item()]
        token_probs.append(p.item())        # <eos>의 확률도 신뢰도에 포함
        if token == EOS_TOKEN:
            break
        chars.append(token)
        dec_in = torch.cat([dec_in, idx.unsqueeze(0)], dim=1)
    if not token_probs:
        return '', 0.0, []
    log_mean = sum(math.log(max(p, 1e-12)) for p in token_probs) / len(token_probs)
    return ''.join(chars), math.exp(log_mean), token_probs


import math

print(f'{"입력":<22}{"생성":<12}{"신뢰도":>8}{"최저 토큰 확률":>16}  정답 여부')
print('-' * 72)
for i in range(3000, 3010):
    gen, conf, ps = predict_with_confidence(dc_model, dc_src[i], dc_sv, dc_tv, device)
    mark = '정답' if gen == dc_tgt[i] else f'오답({dc_tgt[i]})'
    print(f'{dc_src[i]:<22}{gen:<12}{conf * 100:>7.1f}%{min(ps) * 100:>15.1f}%  {mark}')

입력                    생성               신뢰도        최저 토큰 확률  정답 여부
------------------------------------------------------------------------
B@GbQb}$aG%yQ[27 July 2000f|u#pPvR-U2000-7-27     100.0%           99.9%  정답
w3E^3Z03zh9){20 Sep 2042LLz2042-9-30      97.2%           75.7%  오답(2042-9-20)
U3k:Dj6bJune 27, 2029bzs$uE?je7P(#3c|92029-6-27     100.0%           99.9%  정답
.tc_Oqhny7$?Vv/2A(UkWe>]Jan 25, 1949&O1949-1-25     100.0%          100.0%  정답
j<#b:<L]N08/22/19518TgU4?lcn-1951-8-22     100.0%          100.0%  정답
VS{:-%m<*M(nbqV=f1921/11/141921-11-14     99.9%           99.2%  정답
.jPk+25-11-2002fEAOqb6zBW9N+gSr6ExvzPGy92002-11-25    100.0%           99.8%  정답
X&{&#Ka<+!;*{,sb8j1Ik+r2021-05-29B;Q)qG2021-5-2       94.6%           60.9%  오답(2021-5-29)
F_TbW0Ybi19 June 1997eH]1997-6-19     100.0%           99.9%  정답
8b:jZNu.XXB[1La6-=17 Feb 1968,P_0exeH1968-2-17     100.0%          100.0%  정답


In [27]:

# 신뢰도가 정답 여부를 실제로 가려내는지 확인
rows = []
for i in range(3000, 4000):
    gen, conf, ps = predict_with_confidence(dc_model, dc_src[i], dc_sv, dc_tv, device)
    rows.append((gen == dc_tgt[i], conf, min(ps)))

ok = [r for r in rows if r[0]]
ng = [r for r in rows if not r[0]]
print(f'정답 {len(ok)}건, 오답 {len(ng)}건')
print()
print(f'{"구분":<8}{"평균 신뢰도(기하 평균)":>24}{"평균 최저 토큰 확률":>22}')
print('-' * 56)
for name, group in (('정답', ok), ('오답', ng)):
    if group:
        print(f'{name:<8}{sum(r[1] for r in group) / len(group) * 100:>23.1f}%'
              f'{sum(r[2] for r in group) / len(group) * 100:>21.1f}%')

정답 957건, 오답 43건

구분                 평균 신뢰도(기하 평균)           평균 최저 토큰 확률
--------------------------------------------------------
정답                         99.8%                 98.5%
오답                         96.8%                 78.5%


### 풀이 해설 — 연습 문제 10-9

**지문이 요구한 조건은 "생성 결과의 길이에 무관하게"다.** 여기에 답이 들어 있다.

**단순히 확률을 곱하면 안 된다.** 순차 데이터 전체의 확률은 토큰 확률의 곱이고, 각
확률이 1 이하이므로 **길이가 길수록 값이 작아진다.** `1948-4-30`(9글자)과
`2026-2-1`(8글자)의 신뢰도를 나란히 놓을 수 없다. 10-3절 빔 서치에서 다룬 문제와 같다.

**해법은 기하 평균이다.**

```python
log_mean = sum(math.log(p) for p in token_probs) / len(token_probs)
confidence = math.exp(log_mean)
```

로그 확률의 **평균**을 내고 지수를 씌우면 **토큰 하나당 평균 확률**이 된다. 길이로
나누었으므로 길이가 달라도 비교할 수 있고, 값의 범위도 0~1로 분류 모델의 신뢰도와 같은
눈금이 된다.

| 방법 | 성격 |
|---|---|
| 확률의 곱 | 길이에 좌우된다. 비교 불가 |
| **기하 평균** | **길이에 무관한 평균 확률. 무난한 기본값** |
| 산술 평균 | 계산은 쉽지만 확률의 곱셈 구조를 반영하지 못한다 |
| 최저 토큰 확률 | **가장 불안했던 한 자리**를 본다. 보수적 |

날짜 변환은 **한 글자만 틀려도 오답**이므로 최저 토큰 확률도 함께 볼 만하다. 아홉 글자
중 여덟 개를 99%로 확신하고 한 글자만 40%라면 기하 평균은 90%를 넘지만 실제로는
위태롭다.

**★ 신뢰도는 실제로 오답을 걸러 낸다.** 검증 데이터 1,000건의 결과다.

| 구분 | 건수 | 평균 신뢰도(기하 평균) | 평균 최저 토큰 확률 |
|---|---|---|---|
| 정답 | 957 | 99.8% | 98.5% |
| 오답 | 43 | 96.8% | **78.5%** |

**두 지표의 성격 차이가 그대로 드러난다.** 기하 평균은 99.8% 대 96.8%로 3.0%포인트밖에
벌어지지 않지만, **최저 토큰 확률은 98.5% 대 78.5%로 20.0%포인트 벌어진다.** 날짜
변환처럼 아홉 글자 중 한 글자만 틀려도 오답인 문제에서는 **가장 불안했던 한 자리를 보는
쪽이 훨씬 예리하다.** 나머지 여덟 글자를 99.9%로 맞히면 기하 평균은 틀린 한 글자를
희석해 버린다.

생성 예시에서도 같은 모습이 보인다.

```
w3E^3Z03zh9){20 Sep 2042LLz   2042-9-30    신뢰도 97.2%   최저 토큰 확률 75.7%   오답(2042-9-20)
X&{&#...2021-05-29B;Q)qG      2021-5-2     신뢰도 94.6%   최저 토큰 확률 60.9%   오답(2021-5-29)
```

두 오답 모두 **신뢰도만 보면 94~97%로 높지만 최저 토큰 확률은 60~76%로 내려앉는다.**
틀린 자리 하나가 정확히 그 자리다.

**그러므로 임계값은 최저 토큰 확률에 거는 편이 낫다.** 예를 들어 90% 미만을 사람이
확인하도록 하면 오답을 상당수 건져 올리면서 정답을 불필요하게 걸러 내는 일은 적다.

**모델이 자기 오류를 다 아는 것은 아니다.** 오답의 평균 신뢰도가 96.8%나 된다는 것은
**틀리면서도 꽤 확신하고 있다**는 뜻이다. 10-2절 본문이 표 10-7에서 보인 것과 같은
현상이다. 신뢰도는 오답을 완벽히 가려내는 장치가 아니라 **사람이 볼 것의 우선순위를
정해 주는 장치**로 보는 편이 정확하다.

## 연습 문제 10-10 [도전 문제]

In [28]:

# 정렬 시리즈의 네 번째: 수열이 오름차순인지 내림차순인지 정렬되지 않았는지 판정
ORDER_LABELS = ['오름차순', '내림차순', '정렬되지 않음']

order_rng = random.Random(SEED)
order_samples = []
for _ in range(12000):
    nums = [order_rng.randint(1, 1000) for _ in range(10)]
    kind = order_rng.randrange(3)
    if kind == 0:
        nums.sort()
    elif kind == 1:
        nums.sort(reverse=True)
    else:
        # 정렬된 상태로 우연히 만들어지는 경우를 배제
        while sorted(nums) == nums or sorted(nums, reverse=True) == nums:
            order_rng.shuffle(nums)
    order_samples.append((', '.join(str(n) for n in nums), kind))

for text, kind in order_samples[:3]:
    print(f'  {ORDER_LABELS[kind]:<10s} {text}')
print()
print('레이블 분포:', Counter(k for _, k in order_samples))
print('입력 길이 최소/최대:',
      min(len(t) for t, _ in order_samples), max(len(t) for t, _ in order_samples))

  정렬되지 않음    655, 115, 26, 760, 282, 251, 229, 143, 755, 105
  오름차순       31, 33, 90, 96, 224, 433, 559, 605, 759, 914
  오름차순       28, 204, 430, 518, 559, 575, 617, 666, 719, 734

레이블 분포: Counter({2: 4017, 0: 4000, 1: 3983})
입력 길이 최소/최대: 41 49


In [29]:

# 데이터로더 구성([CLS]를 앞에 붙이는 방식은 본문과 같다)
n_tr, n_va = 7680, 1920
order_split = {
    'train': order_samples[:n_tr],
    'valid': order_samples[n_tr:n_tr + n_va],
    'test': order_samples[n_tr + n_va:],
}
order_vocab = Vocab([t for t, _ in order_split['train']], special_tokens)
order_loaders = {
    name: DataLoader(
        DateFormatDataset([t for t, _ in split], [k for _, k in split], order_vocab),
        batch_size=BATCH_SIZE, shuffle=(name == 'train'), collate_fn=collate_fn)
    for name, split in order_split.items()
}
ORDER_MAX_LENGTH = max(len(t) for t, _ in order_samples) + 2
print(f'어휘 {len(order_vocab)}, 최대 길이 {ORDER_MAX_LENGTH}')
print(f'훈련/검증/평가 {len(order_split["train"])} / {len(order_split["valid"])} '
      f'/ {len(order_split["test"])}')

common.set_seed(SEED)
model_order = DateFormatClassifier(
    vocab_size=len(order_vocab), d_model=D_MODEL, num_heads=NUM_HEADS,
    ff_dim=FF_DIM, num_layers=NUM_LAYERS, max_length=ORDER_MAX_LENGTH,
    num_classes=3, dropout=DROPOUT).to(device)
train_classifier(model_order, order_loaders['train'], order_loaders['valid'],
                 name='정렬 여부 판정')
acc_order = test_accuracy(model_order, order_loaders['test'])
print(f'\n테스트셋 정확도 {acc_order:.2f}%')

어휘 15, 최대 길이 51
훈련/검증/평가 7680 / 1920 / 2400
정렬 여부 판정 학습


에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/60       0.5277       0.2539       90.78%     0:01


  6/60       0.1220       0.1123       95.99%     0:09


 12/60       0.0823       0.0815       97.40%     0:18


 18/60       0.0584       0.0895       97.45%     0:27


 24/60       0.0467       0.0703       98.12%     0:36


 30/60       0.0456       0.0445       98.75%     0:45


 36/60       0.0371       0.0621       98.39%     0:54


------------------------------------------------------
최적 30 에포크 · 검증 손실 0.0445 · 전체 학습 시간 0:57 · (조기 종료)

테스트셋 정확도 99.04%


In [30]:

# 어떤 유형에서 틀리는지 확인
order_preds = []
model_order.eval()
with torch.no_grad():
    for start in range(0, len(order_split['test']), 128):
        chunk = order_split['test'][start:start + 128]
        seqs = [torch.LongTensor([CLS_IDX] + order_vocab.encode(t)) for t, _ in chunk]
        src = pad_sequence(seqs, batch_first=True, padding_value=PAD_IDX).to(device)
        order_preds += model_order(src).argmax(dim=1).tolist()

order_true = [k for _, k in order_split['test']]
cm_order = np.zeros((3, 3), dtype=int)
for t, p in zip(order_true, order_preds):
    cm_order[t][p] += 1
print('혼동 행렬 (행: 정답, 열: 예측)')
print(pad('정답 \\ 예측', 14) + ''.join(f'{n:>12s}' for n in ORDER_LABELS))
print('-' * 52)
for i, name in enumerate(ORDER_LABELS):
    print(pad(name, 14) + ''.join(f'{cm_order[i][j]:>12d}' for j in range(3)))

# '거의 정렬된' 입력에서는 어떨까: 오름차순에서 두 자리만 바꾼 입력
near_rng = random.Random(7)
near = []
for _ in range(300):
    nums = sorted(near_rng.randint(1, 1000) for _ in range(10))
    i = near_rng.randrange(9)
    nums[i], nums[i + 1] = nums[i + 1], nums[i]
    near.append(', '.join(str(n) for n in nums))

model_order.eval()
with torch.no_grad():
    seqs = [torch.LongTensor([CLS_IDX] + order_vocab.encode(t)) for t in near]
    src = pad_sequence(seqs, batch_first=True, padding_value=PAD_IDX).to(device)
    near_pred = model_order(src).argmax(dim=1).tolist()
hit = sum(p == 2 for p in near_pred)
print(f'\n이웃한 두 수만 뒤바꾼 입력 300개 중 "정렬되지 않음"으로 맞힌 비율: '
      f'{hit / 300 * 100:.1f}%')

혼동 행렬 (행: 정답, 열: 예측)
정답 \ 예측           오름차순        내림차순     정렬되지 않음
----------------------------------------------------
오름차순               804           0           4
내림차순                 0         777           0
정렬되지 않음           15           4         796

이웃한 두 수만 뒤바꾼 입력 300개 중 "정렬되지 않음"으로 맞힌 비율: 6.0%


### 풀이 해설 — 연습 문제 10-10

**정렬 시리즈에서 이 문제만 성격이 다르다.** 9-7, 9-9, 10-1, 10-17은 **정렬된 수열을
만드는** 생성 문제인데, 10-10은 **정렬 여부를 판정하는** 분류 문제다. 그래서 디코더가
필요 없고 인코더만으로 풀 수 있다.

**데이터를 만들 때 함정이 하나 있다.** '정렬되지 않음' 클래스를 만들려고 무작위로
섞으면, **우연히 정렬된 수열이 나올 수 있다.** 열 개라면 확률이 매우 낮지만 12,000개를
만들면 걸릴 수 있고, 그런 샘플은 레이블이 틀린 학습 데이터가 된다. 이 풀이는
`while`로 걸러 냈다. 본문 10-2절이 형식이 유일하게 결정되지 않는 샘플을 폐기한 것과
같은 종류의 처리다.

**이 문제가 셀프 어텐션에 잘 맞는 이유**를 생각해 보면 재미있다. 정렬 여부 판정은
**이웃한 두 수를 모두 비교**하면 끝난다. `a1 ≤ a2 ≤ ... ≤ a10`인지 확인하는 아홉 번의
비교다. 셀프 어텐션은 모든 토큰 쌍의 관계를 한 번에 계산하므로 이 구조에 잘 맞는다.
순환 신경망이라면 앞의 수를 기억한 채 다음 수와 비교하며 열 단계를 거쳐야 한다.

**반면 [연습 문제 10-1]의 생성 문제는 훨씬 어렵다.** 판정은 '이웃끼리 비교'라는 국소
연산의 합이지만, 정렬해서 **써 내는** 일은 열 개를 모두 견주어 순서를 정하고 그 결과를
기억해야 한다. 같은 '정렬'이라는 말이 붙어 있어도 난도가 크게 다르다. 정확도를 나란히
놓고 보면 그 차이가 드러난다.

**'거의 정렬된' 입력으로 확인하는 것이 이 풀이의 요점이다.** 무작위로 섞인 수열은
어디를 봐도 순서가 어긋나 있어 쉽게 판정된다. 진짜 시험은 **딱 한 자리만 어긋난**
입력이다. 모델이 '전체 분위기'가 아니라 **모든 이웃 쌍을 실제로 비교**하고 있다면 이런
입력도 잡아내야 한다. 정확도가 크게 떨어진다면, 모델이 아홉 번의 비교를 하는 대신
대충의 경향만 보고 있다는 뜻이다.

`[CLS]` 하나의 벡터에 '아홉 번의 비교 결과를 모두 통과했는가'를 담아야 한다는 점도
부담이다. 자릿수가 다른 수(`9`와 `1000`)를 글자 단위로 비교해야 한다는 점까지
생각하면, 사람 눈에 쉬워 보이는 문제가 모델에게는 그렇지 않다는 것을 알 수 있다.


## 연습 문제 10-11 [도전 문제]

In [31]:

# 숫자로 월을 표기하는 네 형식만 대상으로 데이터를 다시 만든다
NUM_ONLY_IDX = [2, 3, 4, 5]          # %m/%d/%Y, %Y/%m/%d, %d-%m-%Y, %Y-%m-%d
NUM_ONLY_FORMATS = [SRC_FORMATS[i] for i in NUM_ONLY_IDX]
print('대상 형식:', NUM_ONLY_FORMATS)

num_dataset = {
    name: [Sample(s.text, NUM_ONLY_IDX.index(s.label), s.clean, s.noise_length)
           for s in split if s.label in NUM_ONLY_IDX]
    for name, split in dataset.items()
}
for name, split in num_dataset.items():
    print(f'  {name}: {len(split)}개')

대상 형식: ['%m/%d/%Y', '%Y/%m/%d', '%d-%m-%Y', '%Y-%m-%d']
  train: 5120개
  valid: 1280개
  test: 1600개


In [32]:

# 두 자리 숫자와 구분자로 이루어진 토큰 집합
#   두 자리로 묶을 수 없는 홀수 자리 숫자와 어휘에 없는 노이즈의 처리 방법을 먼저 정한다
TWO_DIGIT = [f'{i:02d}' for i in range(100)]      # '00' ~ '99'
SEPARATORS = ['/', '-']
NUM_SPECIALS = {'[CLS]': 0, '[PAD]': 1, '[UNK]': 2, '[ODD]': 3}


class TwoDigitVocab:
    """두 자리 숫자, 구분자, 특수 토큰으로 이루어진 어휘 사전

    - 숫자는 왼쪽부터 두 자리씩 묶는다.
    - 묶고 남은 한 자리는 [ODD] 토큰으로 바꾼다(홀수 개로 이어진 숫자 처리).
    - 구분자가 아닌 글자는 모두 [UNK]로 바꾼다(노이즈 처리).
    """

    def __init__(self):
        self.vocab = dict(NUM_SPECIALS)
        for t in TWO_DIGIT + SEPARATORS:
            self.vocab[t] = len(self.vocab)
        self.itos = {v: k for k, v in self.vocab.items()}

    def encode(self, text):
        ids, i = [], 0
        while i < len(text):
            ch = text[i]
            if ch.isdigit():
                run = 0
                while i + run < len(text) and text[i + run].isdigit():
                    run += 1
                for k in range(0, run - 1, 2):
                    ids.append(self.vocab[text[i + k:i + k + 2]])
                if run % 2 == 1:                       # 짝을 못 이룬 마지막 한 자리
                    ids.append(self.vocab['[ODD]'])
                i += run
            elif ch in SEPARATORS:
                ids.append(self.vocab[ch])
                i += 1
            else:
                ids.append(self.vocab['[UNK]'])        # 노이즈
                i += 1
        return ids

    def __len__(self):
        return len(self.vocab)


num_vocab = TwoDigitVocab()
sample = num_dataset['test'][0]
print(f'입력: {sample.text!r}')
print(f'날짜: {sample.clean!r}')
print('토큰:', [num_vocab.itos[i] for i in num_vocab.encode(sample.text)])
print()
lens_char = [len(s.text) for s in num_dataset['test']]
lens_tok = [len(num_vocab.encode(s.text)) for s in num_dataset['test']]
print(f'글자 단위 어휘 {len(vocab)}, 평균 길이 {sum(lens_char) / len(lens_char):.1f}')
print(f'두 자리 단위 어휘 {len(num_vocab)}, 평균 길이 {sum(lens_tok) / len(lens_tok):.1f}')

입력: 'o!z/941AduDvLw;TDEK}1uO^{;w11/04/2043'
날짜: '11/04/2043'
토큰: ['[UNK]', '[UNK]', '[UNK]', '/', '94', '[ODD]', '[UNK]', '[UNK]', '[UNK]', '[UNK]', '[UNK]', '[UNK]', '[UNK]', '[UNK]', '[UNK]', '[UNK]', '[UNK]', '[UNK]', '[UNK]', '[ODD]', '[UNK]', '[UNK]', '[UNK]', '[UNK]', '[UNK]', '[UNK]', '11', '/', '04', '/', '20', '43']

글자 단위 어휘 93, 평균 길이 32.5
두 자리 단위 어휘 106, 평균 길이 28.3


In [33]:

# 두 가지 토큰화로 각각 학습해 비교(대상 형식 네 개로 통일)
class NumDataset(Dataset):
    def __init__(self, samples, vc):
        self.items = [([vc.vocab.get('[CLS]', CLS_IDX)] + vc.encode(s.text), s.label)
                      for s in samples]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


def num_collate(batch):
    seqs, labels = zip(*batch)
    x = pad_sequence([torch.LongTensor(s) for s in seqs],
                     batch_first=True, padding_value=1)      # [PAD] = 1
    return x, torch.LongTensor(labels)


num_loaders = {
    name: DataLoader(NumDataset(split, num_vocab), batch_size=BATCH_SIZE,
                     shuffle=(name == 'train'), collate_fn=num_collate)
    for name, split in num_dataset.items()
}

# 비교군: 같은 네 형식을 글자 단위로
char_loaders = {
    name: DataLoader(
        DateFormatDataset([s.text for s in split], [s.label for s in split], vocab),
        batch_size=BATCH_SIZE, shuffle=(name == 'train'), collate_fn=collate_fn)
    for name, split in num_dataset.items()
}


class NumFormatClassifier(DateFormatClassifier):
    """패딩 번호만 다른 분류기([PAD] = 1)"""

    def forward(self, source):
        pad_mask = (source == 1)
        x = self.dropout(self.pos_encoding(self.embedding(source)))
        output = self.encoder(x, src_key_padding_mask=pad_mask)
        return self.classifier(output[:, 0, :])


for tag, loaders_x, vsize, cls in (
        ('글자 단위', char_loaders, len(vocab), DateFormatClassifier),
        ('두 자리 숫자 단위', num_loaders, len(num_vocab), NumFormatClassifier)):
    common.set_seed(SEED)
    m = cls(vocab_size=vsize, d_model=D_MODEL, num_heads=NUM_HEADS, ff_dim=FF_DIM,
            num_layers=NUM_LAYERS, max_length=MAX_LENGTH, num_classes=4,
            dropout=DROPOUT).to(device)
    train_classifier(m, loaders_x['train'], loaders_x['valid'], name=tag)
    print(f'{tag} 테스트 정확도 {test_accuracy(m, loaders_x["test"]):.2f}%\n')

글자 단위 학습


에포크    훈련 손실    검증 손실    정확도(%)     시간
  1/60       0.8828       0.7363       49.06%     0:01


  6/60       0.7073       0.7244       53.12%     0:06


 12/60       0.6036       0.6578       66.09%     0:12


 18/60       0.4246       0.4378       81.56%     0:18


...

 24/60       0.1266       0.1590       94.84%     0:34


 30/60       0.0661       0.1030       97.58%     0:41


 36/60       0.0575       0.0634       98.44%     0:49


 42/60       0.0389       0.0743       98.05%     0:59


------------------------------------------------------
최적 36 에포크 · 검증 손실 0.0634 · 전체 학습 시간 1:03 · (조기 종료)
두 자리 숫자 단위 테스트 정확도 98.31%



### 풀이 해설 — 연습 문제 10-11

**지문이 "먼저 결정해야 한다"고 못 박은 두 가지가 이 문제의 핵심이다.**

**하나, 어휘에 없는 노이즈를 어떻게 할 것인가.** 두 자리 숫자와 구분자만 토큰으로
두면 알파벳과 특수문자는 전부 어휘 밖이다. 이 풀이는 모두 `[UNK]` 하나로 보냈다.
그 결과 **노이즈의 내용이 사라지고 '노이즈가 몇 개 있었다'는 정보만 남는다.** 형식
분류에는 노이즈의 내용이 필요 없으므로 손해가 아니며, 오히려 **모델이 노이즈에
속을 여지를 줄인다.**

**둘, 홀수 개로 이어진 숫자를 어떻게 할 것인가.** 이쪽이 더 까다롭다.
`1938`은 `19` + `38`로 깔끔히 나뉘지만, 노이즈에 숫자가 섞이면 `...9` + `1938`처럼
**다섯 자리가 이어지는** 상황이 생긴다. 선택지는 여럿이다.

| 방법 | 성격 |
|---|---|
| **`[ODD]` 토큰으로 바꾼다**(이 풀이) | 자릿수가 어긋났다는 사실 자체를 신호로 남긴다 |
| 남은 한 자리를 버린다 | 길이는 짧아지지만 날짜의 첫 자리를 잃을 수 있다 |
| 한 자리 숫자 토큰 10개를 추가한다 | 어휘가 늘지만 정보 손실이 없다 |
| 오른쪽부터 묶는다 | 날짜 쪽 정렬이 맞을 때도 있고 어긋날 때도 있다 |

`[ODD]`를 택한 이유는 **10-6에서 세운 가설과 이어지기 때문이다.** 오분류의 원인으로
'노이즈의 숫자가 날짜에 달라붙어 자릿수 경계를 흐린다'를 지목했는데, 두 자리 묶음
토큰화에서는 그 현상이 **`[ODD]` 토큰의 등장**이라는 눈에 보이는 형태로 드러난다.

**토큰화 단위가 미치는 영향은 두 방향으로 갈린다.**

**유리한 쪽** — 순차 데이터가 절반 이하로 짧아진다. 셀프 어텐션의 계산량은 길이의
제곱에 비례하므로 계산이 크게 준다. 더 중요한 것은 **`19`, `38` 같은 두 자리 수가 하나의
토큰이 되어, 모델이 자릿수를 세는 일을 하지 않아도 된다**는 점이다. `%m/%d/%Y`와
`%Y/%m/%d`를 가르는 단서가 '숫자 덩어리의 길이'인데, 이제 토큰 개수로 바로 드러난다.

**불리한 쪽** — 어휘 사전이 글자 30개 안팎에서 100개 이상으로 늘고, 각 토큰의 등장
빈도가 줄어 **임베딩을 학습할 데이터가 토큰당 적어진다.** 그리고 결정적으로,
**토큰화 단계에서 이미 한 번의 해석이 일어난다.** 왼쪽부터 두 자리씩 묶는다는 규칙 자체가
사람이 넣은 가정이고, 그 가정이 어긋나는 입력(노이즈 숫자가 붙은 경우)에서는 **모델이
보기도 전에 정보가 뭉개진다.**

**이것이 토큰화 단위 선택의 본질이다.** 토큰을 크게 잡으면 모델의 일이 줄지만, 대신
**사람이 미리 내린 판단이 데이터에 박힌다.** 잘못 박히면 모델이 고칠 방법이 없다.
[연습 문제 10-16]에서 반대 방향(단어 → 글자)을 다루는데, 결론은 같은 축 위에 있다.


---

## 정리

10-2절의 여섯 문제는 인코더만 사용하는 트랜스포머를 세 방향으로 살펴본다.

- **분석(10-6)** — 혼동 행렬에서 출발해 가설을 세우고 데이터로 검증한다. 모델을
  고치는 대신 **모델이 무엇에 속는지 알아내는** 훈련이다.
- **구조 변형(10-7, 10-8, 10-10)** — `[CLS]` 대신 평균 풀링, 토큰마다 분류, 세 갈래
  분류. 인코더 트랜스포머를 **분류기 부분만 바꿔** 여러 문제에 붙여 본다.
- **입력 표현(10-11)과 출력 해석(10-9)** — 토큰화 단위를 바꾸거나, 결과에 신뢰도를
  붙인다. 모델 바깥에서 할 수 있는 일들이다.

[연습 문제 10-10]은 9-7에서 시작한 정렬 시리즈의 네 번째 문제로, 유일하게 **생성이
아닌 분류** 문제다.
